In [2]:
import tensorflow as tf
print(f"Tensorflow Versi: \n{tf.__version__}")

Tensorflow Versi: 
2.21.0


abs = (Jarak sebenarnya) mengubah nilai negatif menjadi positif dan positif menjadi positif -> ini MAE
**2 = (kasih hukuman ekstra ke error besar) -> ini MAE

In [3]:
# Cell 1 - Custom Layer
class DoubleLayer(tf.keras.layers.Layer):
    def call(self,inputs):
        return inputs * 2

layer = DoubleLayer()
x = tf.constant([
    [1.0],
    [2.0],
    [3.0]
])
hasil = layer(x)
print(f"Input: \n{x.numpy()}")
print(f"output: \n{hasil.numpy()}")

Input: 
[[1.]
 [2.]
 [3.]]
output: 
[[2.]
 [4.]
 [6.]]


In [4]:
# Cell 2 - Custom Layer dengan Weight Sendiri
class MyDense(
    tf.keras.layers.Layer
):
    def build(self,input_shape): # ada inputs dan ada inputs
        self.w = self.add_weight(
            shape=(input_shape[-1],1),
            initializer="random_normal"
        )

        self.b = self.add_weight(
            shape=(1,),
            initializer="zeros"
        )

    def call(
            self,
            inputs
    ):
        
        return (
            tf.matmul(inputs,self.w) + self.b
        )
layer = MyDense()

x = tf.constant([
    [1.0],
    [2.0],
    [3.0]
])
hasil = layer(x)
print(f"Weight: \n{layer.w.numpy()}")
print(f"Bias: \n{layer.b.numpy()}")
print(f"output: \n{hasil.numpy()}")


Weight: 
[[-0.0543779]]
Bias: 
[0.]
output: 
[[-0.0543779 ]
 [-0.1087558 ]
 [-0.16313371]]


In [5]:
# Cell 3 - Custom Model
class MyModel(
    tf.keras.Model
):
    
    def __init__(self):
        super().__init__()

        self.dense1= tf.keras.layers.Dense(8,activation="relu")
        self.dense2 = tf.keras.layers.Dense(1)
    def call(self,inputs):
        x = self.dense1(inputs)
        return self.dense2(x)
    
model = MyModel()

x = tf.constant([
    [1.0],
    [2.0],
    [3.0]
])
hasil = model(x)
print(f"hasil: \n{hasil.numpy()}")

hasil: 
[[-0.16762279]
 [-0.33524558]
 [-0.5028683 ]]


In [6]:
# Cell 4 - Custom Loss Function
def my_mse(y_true,y_pred):
    return tf.reduce_mean(
        (y_true - y_pred) ** 2
    )
y_true = tf.constant([
    [10.0],
    [20.0]
]) 
y_pred = tf.constant([
    [12.0],
    [18.0]
])
loss = my_mse(y_true,y_pred)
print(loss.numpy())

4.0


In [7]:
# Cell 5 - Custom Metric
def my_mse(y_true,y_pred):
    return tf.reduce_mean(
        tf.abs(y_true - y_pred) # ini adalah mae
    )
y_true = tf.constant([
    [10.0],
    [20.0]
])
y_pred = tf.constant([
    [12.0],
    [18.0]
])
hasil = my_mse(y_true,y_pred)
print(f"Hasil: \n{hasil.numpy()}")

Hasil: 
2.0


In [8]:
# Cell 6 - Dataset dan Model
import tensorflow as tf
x_train = tf.constant([
    [1.0],
    [2.0],
    [3.0],
    [4.0]
])

y_train = tf.constant([
    [3.0],
    [5.0],
    [7.0],
    [9.0]
])
dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)
dataset = dataset.batch(2)
model = MyModel()
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.01
)
print(f"Dataset siap")

Dataset siap


In [9]:
#Cell 7 - Custom Training Loop
for epoch in range(100):
    total_loss = 0

    for x_batch, y_batch in dataset:
        with tf.GradientTape() as tape:
            prediksi = model(x_batch)

            loss = tf.reduce_mean(
                (y_batch - prediksi) ** 2
            )

        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        total_loss += loss
    if epoch % 10 == 0:
        print(
            f"Epoch {epoch}",
            f"Loss {loss}"
        )


Epoch 0 Loss 55.82482147216797
Epoch 10 Loss 36.1491813659668
Epoch 20 Loss 16.3449649810791
Epoch 30 Loss 3.263965129852295
Epoch 40 Loss 0.0879397913813591
Epoch 50 Loss 0.011738094501197338
Epoch 60 Loss 0.02940550446510315
Epoch 70 Loss 0.044118285179138184
Epoch 80 Loss 0.0340939462184906
Epoch 90 Loss 0.02925625443458557


In [10]:
# Cell 8 - Melihat Semua Weight Model
for variable in model.trainable_variables:
    print(variable.name)
    print(variable.numpy())
    print()

kernel
[[ 1.0619453  -0.70561296 -0.03765714 -0.06944761 -0.4031776  -0.7991873
  -0.37965605  1.057043  ]]

bias
[ 0.60784286  0.          0.         -0.13631257  0.          0.
  0.          0.6382822 ]

kernel
[[ 0.91187507]
 [ 0.03857774]
 [ 0.26204026]
 [-0.6328757 ]
 [-0.39272666]
 [ 0.53678584]
 [ 0.08409834]
 [ 0.78373235]]

bias
[0.52683634]



In [11]:
prediksi = model(
    tf.constant([
        [10.0],
        [20.0],
        [30.0]
    ])
)
print(f"Prediksi: \n{prediksi.numpy()}")

Prediksi: 
[[19.549358]
 [37.51736 ]
 [55.485363]]


In [12]:
# Cell 10 - Menyimpan dan Memuat Weight
model.save_weights(
    "my_weight.weights.h5"
)
print("Weight disimpan")
model_baru = MyModel()

dummy = tf.constant([
    [1.0]
])
model_baru(dummy)
model_baru.load_weights(
    "my_weight.weights.h5"
)
hasil = model_baru(
    tf.constant([
        [10.0]
    ])
)
print(f"hasil: \n{hasil.numpy()}")

Weight disimpan
hasil: 
[[19.549358]]
